In [0]:
from pyspark.sql.functions import *
from delta.tables import *

spark.sql("CREATE CATALOG IF NOT EXISTS retail_project")
spark.sql("CREATE SCHEMA IF NOT EXISTS retail_project.retail_gold")

spark.sql("USE CATALOG retail_project")
spark.sql("USE SCHEMA retail_gold")

DataFrame[]

In [0]:
orders = spark.table("retail_project.retail_silver.orders")
customers = spark.table("retail_project.retail_silver.customer")
products = spark.table("retail_project.retail_silver.products")
payments = spark.table("retail_project.retail_silver.payments")
returns = spark.table("retail_project.retail_silver.returns")
shipments = spark.table("retail_project.retail_silver.shipments")
store = spark.table("retail_project.retail_silver.store")
inventory = spark.table("retail_project.retail_silver.inventory")

In [0]:
customers = customers.withColumnRenamed("city","customer_city")

store = store.withColumnRenamed("city","store_city")

products = products.withColumnRenamed("supplier_city","product_supplier_city")
products = products.withColumnRenamed("currency","product_currency")

payments = payments.withColumnRenamed("currency","payment_currency")

shipments = shipments.withColumnRenamed("warehouse_city","shipment_city")
shipments = shipments.withColumnRenamed("shipping_cost","shipment_cost")

inventory = inventory.withColumnRenamed("stock_qty","inventory_stock_qty")

In [0]:
gold_base = (
    orders
    .join(customers, "order_id", "left")
    .join(payments, "order_id", "left")
    .join(shipments, "order_id", "left")
    .join(returns, "order_id", "left")
    .join(products, "order_id", "left")
    .join(inventory, "order_id", "left")
    .join(store, "order_id", "left")
)

In [0]:
orders = orders.dropDuplicates(["order_id"])
customer = customers.dropDuplicates(["order_id"])
payments = payments.dropDuplicates(["order_id"])
shipments = shipments.dropDuplicates(["order_id"])
returns = returns.dropDuplicates(["order_id"])
products = products.dropDuplicates(["order_id"])
inventory = inventory.dropDuplicates(["order_id"])
store = store.dropDuplicates(["order_id"])

In [0]:
gold_base.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true")\
.saveAsTable("retail_project.retail_gold.master_retail")

In [0]:
master = spark.table("retail_project.retail_gold.master_retail")

### DIM CUSTOMER (SCD2)

In [0]:
dim_customer_src = master.select(
    "customer_id",
    "customer_full_name",
    "gender",
    "customer_city",
    "state",
    "country",
    "tier",
    "customer_status"
).dropDuplicates()

In [0]:
dim_customer_src = dim_customer_src \
.withColumn("effective_date", current_timestamp()) \
.withColumn("end_date", lit(None).cast("timestamp")) \
.withColumn("is_current", lit(1))

In [0]:
dim_customer_src.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true")\
.saveAsTable("retail_project.retail_gold.dim_customer")

In [0]:
dim_customer = DeltaTable.forName(spark, "retail_project.retail_gold.dim_customer")

dim_customer.alias("t").merge(
    dim_customer_src.alias("s"),
    "t.customer_id = s.customer_id AND t.is_current = 1"
).whenMatchedUpdate(
    condition = """
    t.customer_full_name <> s.customer_full_name OR
    t.customer_city <> s.customer_city OR
    t.tier <> s.tier
    """,
    set = {
        "end_date": "current_timestamp()",
        "is_current": "0"
    }
).whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### DIM PRODUCT (SCD2)

In [0]:
dim_product_src = master.select(
    "product_id",
    "product_name",
    "category",
    "brand",
    "supplier_name",
    "product_supplier_city",
    "discounted_price",
    "final_price_with_tax",
    "product_popularity"
).dropDuplicates()

In [0]:
dim_product_src = dim_product_src \
.withColumn("effective_date", current_timestamp()) \
.withColumn("end_date", lit(None).cast("timestamp")) \
.withColumn("is_current", lit(1))

In [0]:
dim_product_src.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true")\
.saveAsTable("retail_project.retail_gold.dim_product")

In [0]:
dim_product = DeltaTable.forName(spark, "retail_project.retail_gold.dim_product")

dim_product.alias("t").merge(
    dim_product_src.alias("s"),
    "t.product_id = s.product_id AND t.is_current = 1"
).whenMatchedUpdate(
    condition = """
    t.discounted_price <> s.discounted_price OR
    t.product_popularity <> s.product_popularity
    """,
    set = {
        "end_date": "current_timestamp()",
        "is_current": "0"
    }
).whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### DIM STORE (SCD2)

In [0]:
dim_store_src = master.select(
    "store_id",
    "store_name",
    "store_city",
    "store_type",
    "store_size_category",
    "manager_name",
    "experience_years"
).dropDuplicates()

In [0]:
dim_store_src = dim_store_src \
.withColumn("effective_date", current_timestamp()) \
.withColumn("end_date", lit(None).cast("timestamp")) \
.withColumn("is_current", lit(1))

In [0]:
dim_store_src.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true")\
.saveAsTable("retail_project.retail_gold.dim_store")

In [0]:
dim_store = DeltaTable.forName(spark, "retail_project.retail_gold.dim_store")

dim_store.alias("t").merge(
    dim_store_src.alias("s"),
    "t.store_id = s.store_id AND t.is_current = 1"
).whenMatchedUpdate(
    condition = """
        t.store_size_category <> s.store_size_category OR
        t.experience_years <> s.experience_years OR
        t.store_city <> s.store_city
    """,
    set = {
        "end_date": "current_timestamp()",
        "is_current": "0"
    }
).whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### FACT SALES

In [0]:
fact_sales_src = master.select(
    "order_id",
    "customer_id",
    "product_id",
    "store_id",
    "payment_id",
    "shipment_id",
    "order_date",
    "quantity",
    "unit_price",
    "calculated_order_value",
    "shipping_cost",
    "delivery_days"
).dropDuplicates(["order_id"])

In [0]:
fact_sales_src.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true")\
.saveAsTable("retail_project.retail_gold.fact_sales")

In [0]:
fact_sales = DeltaTable.forName(spark, "retail_project.retail_gold.fact_sales")

fact_sales.alias("t").merge(
    fact_sales_src.alias("s"),
    "t.order_id = s.order_id"
).whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### FACT RETURNS

In [0]:
fact_returns_src = master.select(
    "return_id",
    "order_id",
    "customer_id",
    "product_id",
    "refund_amount",
    "return_processing_time"
).dropDuplicates(["return_id"])

In [0]:
fact_returns_src.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true")\
.saveAsTable("retail_project.retail_gold.fact_returns")

In [0]:
fact_returns = DeltaTable.forName(spark, "retail_project.retail_gold.fact_returns")

fact_returns.alias("t").merge(
    fact_returns_src.alias("s"),
    "t.return_id = s.return_id"
).whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]